# Cluster Exploration

Interactive exploration of stock clusters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 1. Load Data

In [ ]:
# Load clustered data
data_dir = Path('../data')

# Choose clustering method: 'kmeans', 'gmm', 'pca_kmeans', or 'hdbscan'
method = 'kmeans'

file_path = data_dir / f'clustered_{method}.csv'
data = pd.read_csv(file_path, parse_dates=['Date'])
data = data.set_index(['Date', 'Ticker']).sort_index()

print(f"Loaded {len(data):,} observations")
print(f"Date range: {data.index.get_level_values('Date').min()} to {data.index.get_level_values('Date').max()}")
print(f"Unique tickers: {data.index.get_level_values('Ticker').nunique()}")
print(f"Number of clusters: {data['cluster'].nunique()}")

## 2. Overall Cluster Distribution

In [ ]:
# Cluster sizes
cluster_counts = data['cluster'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
cluster_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Total Observations per Cluster', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Cluster ID')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
cluster_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Cluster Distribution', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("\nCluster sizes:")
for cluster_id, count in cluster_counts.items():
    pct = count / len(data) * 100
    print(f"  Cluster {cluster_id}: {count:>6,} observations ({pct:>5.1f}%)")

## 3. Cluster Evolution Over Time

In [ ]:
# How cluster sizes change over time
cluster_by_date = data.groupby(['Date', 'cluster']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
cluster_by_date.plot(ax=ax, linewidth=2, marker='o', markersize=4)
ax.set_title('Cluster Evolution Over Time', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Number of Stocks', fontsize=12)
ax.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Cluster Characteristics

In [ ]:
# Average feature values by cluster
key_features = ['return_1m', 'return_3m', 'return_6m', 'rsi', 'beta_mkt', 'beta_smb', 'beta_hml', 'dv_rank']

cluster_profiles = data.groupby('cluster')[key_features].mean()

print("Average feature values by cluster:")
print(cluster_profiles.to_string(float_format=lambda x: f'{x:>8.3f}'))

In [ ]:
# Heatmap of cluster characteristics
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(cluster_profiles.T, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=ax, cbar_kws={'label': 'Value'})
ax.set_title('Cluster Feature Profiles', fontsize=16, fontweight='bold')
ax.set_xlabel('Cluster ID', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Return Distributions by Cluster

In [ ]:
# Box plots of returns by cluster
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, period in enumerate(['return_1m', 'return_3m', 'return_6m']):
    data.boxplot(column=period, by='cluster', ax=axes[idx])
    axes[idx].set_title(f'{period.replace("_", " ").title()} Distribution by Cluster', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Cluster ID')
    axes[idx].set_ylabel('Return')
    axes[idx].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.show()

## 6. Cluster Stability (Transitions)

In [ ]:
# Get last two months
dates = sorted(data.index.get_level_values('Date').unique())
date1, date2 = dates[-2], dates[-1]

df1 = data.xs(date1, level='Date')
df2 = data.xs(date2, level='Date')

# Find common tickers
common_tickers = df1.index.intersection(df2.index)

print(f"Transition analysis: {date1.strftime('%Y-%m')} → {date2.strftime('%Y-%m')}")
print(f"Tracking {len(common_tickers)} common stocks\n")

# Build transition matrix
transition_counts = pd.crosstab(
    df1.loc[common_tickers, 'cluster'],
    df2.loc[common_tickers, 'cluster'],
)

transition_pct = pd.crosstab(
    df1.loc[common_tickers, 'cluster'],
    df2.loc[common_tickers, 'cluster'],
    normalize='index'
) * 100

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Counts
sns.heatmap(transition_counts, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title(f'Transition Counts\n{date1.strftime("%Y-%m")} → {date2.strftime("%Y-%m")}', fontsize=14, fontweight='bold')
axes[0].set_xlabel(f'Cluster at {date2.strftime("%Y-%m")}', fontsize=11)
axes[0].set_ylabel(f'Cluster at {date1.strftime("%Y-%m")}', fontsize=11)

# Percentages
sns.heatmap(transition_pct, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1], cbar_kws={'label': 'Percentage'})
axes[1].set_title(f'Transition Percentages\n{date1.strftime("%Y-%m")} → {date2.strftime("%Y-%m")}', fontsize=14, fontweight='bold')
axes[1].set_xlabel(f'Cluster at {date2.strftime("%Y-%m")}', fontsize=11)
axes[1].set_ylabel(f'Cluster at {date1.strftime("%Y-%m")}', fontsize=11)

plt.tight_layout()
plt.show()

# Calculate stability
stayed_same = sum(
    df1.loc[ticker, 'cluster'] == df2.loc[ticker, 'cluster'] 
    for ticker in common_tickers
)
stability_pct = stayed_same / len(common_tickers) * 100
print(f"\n✓ Stability: {stability_pct:.1f}% of stocks stayed in the same cluster")

## 7. Explore Specific Month

In [ ]:
# Choose a date to explore (default: most recent)
target_date = dates[-1]  # Change this to explore different months

month_data = data.xs(target_date, level='Date')

print(f"Exploring: {target_date.strftime('%Y-%m')}")
print(f"Total stocks: {len(month_data)}")
print("\n" + "="*80)

for cluster_id in sorted(month_data['cluster'].unique()):
    cluster_stocks = month_data[month_data['cluster'] == cluster_id]
    
    print(f"\nCluster {cluster_id} - {len(cluster_stocks)} stocks")
    print("-" * 80)
    
    # Sample tickers
    sample_tickers = cluster_stocks.index.tolist()[:15]
    print(f"Sample tickers: {', '.join(sample_tickers)}")
    if len(cluster_stocks) > 15:
        print(f"... and {len(cluster_stocks) - 15} more")
    
    # Key stats
    print(f"\nKey statistics:")
    print(f"  1-month return:   {cluster_stocks['return_1m'].mean():>7.2%}  (median: {cluster_stocks['return_1m'].median():>7.2%})")
    print(f"  3-month return:   {cluster_stocks['return_3m'].mean():>7.2%}  (median: {cluster_stocks['return_3m'].median():>7.2%})")
    print(f"  6-month return:   {cluster_stocks['return_6m'].mean():>7.2%}  (median: {cluster_stocks['return_6m'].median():>7.2%})")
    print(f"  RSI:              {cluster_stocks['rsi'].mean():>7.2f}  (median: {cluster_stocks['rsi'].median():>7.2f})")
    print(f"  Market beta:      {cluster_stocks['beta_mkt'].mean():>7.2f}  (median: {cluster_stocks['beta_mkt'].median():>7.2f})")

In [ ]:
# Show full list of tickers for a specific cluster
cluster_to_show = 0  # Change this to see different clusters

cluster_stocks = month_data[month_data['cluster'] == cluster_to_show]
print(f"All tickers in Cluster {cluster_to_show} ({target_date.strftime('%Y-%m')}):")
print("\n".join([f"  {ticker}" for ticker in sorted(cluster_stocks.index)]))

## 8. Compare Clusters on Specific Features

In [ ]:
# Scatter plot: Beta vs Returns
fig, ax = plt.subplots(figsize=(12, 7))

for cluster_id in sorted(month_data['cluster'].unique()):
    cluster_data = month_data[month_data['cluster'] == cluster_id]
    ax.scatter(
        cluster_data['beta_mkt'],
        cluster_data['return_3m'],
        label=f'Cluster {cluster_id}',
        alpha=0.6,
        s=50
    )

ax.set_xlabel('Market Beta', fontsize=12)
ax.set_ylabel('3-Month Return', fontsize=12)
ax.set_title(f'Beta vs Returns by Cluster ({target_date.strftime("%Y-%m")})', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: RSI vs Returns
fig, ax = plt.subplots(figsize=(12, 7))

for cluster_id in sorted(month_data['cluster'].unique()):
    cluster_data = month_data[month_data['cluster'] == cluster_id]
    ax.scatter(
        cluster_data['rsi'],
        cluster_data['return_1m'],
        label=f'Cluster {cluster_id}',
        alpha=0.6,
        s=50
    )

ax.set_xlabel('RSI', fontsize=12)
ax.set_ylabel('1-Month Return', fontsize=12)
ax.set_title(f'RSI vs Returns by Cluster ({target_date.strftime("%Y-%m")})', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary Statistics Table

In [ ]:
# Create comprehensive summary table
summary = month_data.groupby('cluster').agg({
    'return_1m': ['mean', 'median', 'std'],
    'return_3m': ['mean', 'median', 'std'],
    'rsi': ['mean', 'median'],
    'beta_mkt': ['mean', 'median'],
    'dv_rank': ['mean', 'median'],
})

summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
print(f"\nSummary Statistics by Cluster ({target_date.strftime('%Y-%m')})")
print("="*80)
print(summary.to_string(float_format=lambda x: f'{x:>8.4f}'))